# 03 — Simulation des moyens pompiers (SDIS) — DONNÉES SIMULÉES

> **⚠️ DISCLAIMER — DONNÉES SIMULÉES.** Il n'existe **aucune API publique temps réel** pour la position et la disponibilité des moyens SDIS (pompiers). Les données ci-dessous sont **synthétiques**, mais réalistes (casernes réelles du Sud-Est, statuts plausibles), et servent uniquement à la démo. Elles sont clairement étiquetées `is_simulated = True`.

**Objectif** : générer une table `live_firefighter_units` (état dispo/engagé + position) et, en option, un **fallback aéronefs simulés** si OpenSky ne renvoie aucun bombardier hors saison.

In [ ]:
# Cell: Casernes / bases de référence (Sud-Est) — SIMULÉ
import pandas as pd, random
random.seed(83)
stations = [
    {"unit_id":"SDIS83-CIS-Hyeres",       "dept":"83","lat":43.121,"lon":6.128,"unit_type":"CCF"},
    {"unit_id":"SDIS83-CIS-Draguignan",   "dept":"83","lat":43.540,"lon":6.466,"unit_type":"CCF"},
    {"unit_id":"SDIS13-CIS-Aix",          "dept":"13","lat":43.529,"lon":5.447,"unit_type":"CCF"},
    {"unit_id":"SDIS30-CIS-Nimes",        "dept":"30","lat":43.836,"lon":4.360,"unit_type":"CCF"},
    {"unit_id":"SDIS06-CIS-Grasse",       "dept":"06","lat":43.659,"lon":6.922,"unit_type":"CCF"},
    {"unit_id":"SDIS34-CIS-Beziers",      "dept":"34","lat":43.344,"lon":3.215,"unit_type":"CCF"},
    {"unit_id":"BASE-Nimes-Garons",       "dept":"30","lat":43.757,"lon":4.416,"unit_type":"AERIEN"},
]

In [ ]:
# Cell: Génération de l'état live des unités (dispo/engagé + léger déplacement) — SIMULÉ
from datetime import datetime, timezone
rows = []
for s in stations:
    status = random.choice(["available","available","available","engaged"])
    rows.append({**s,
        "cur_lat": s["lat"] + random.uniform(-0.05,0.05),
        "cur_lon": s["lon"] + random.uniform(-0.05,0.05),
        "status": status,
        "ingest_ts": datetime.now(timezone.utc).isoformat(),
        "is_simulated": True})
df = pd.DataFrame(rows)
spark.createDataFrame(df).write.format("delta").mode("overwrite").saveAsTable("live_firefighter_units")
print("Table 'live_firefighter_units' (SIMULÉE) écrite. Ré-exécute pour rafraîchir statuts/positions.")
df

In [ ]:
# Cell (optionnel): Fallback aéronefs simulés si OpenSky ne renvoie aucun bombardier
sim_aircraft = pd.DataFrame([
  {"icao24":"SIM001","callsign":"PELICAN32","aircraft_role":"PELICAN","latitude":43.30,"longitude":6.40,"on_ground":False,"is_simulated":True},
  {"icao24":"SIM002","callsign":"MILAN73", "aircraft_role":"MILAN",  "latitude":43.60,"longitude":4.90,"on_ground":False,"is_simulated":True},
  {"icao24":"SIM003","callsign":"DRAGON83","aircraft_role":"DRAGON", "latitude":43.15,"longitude":6.15,"on_ground":False,"is_simulated":True},
])
sim_aircraft["ingest_ts"] = datetime.now(timezone.utc).isoformat()
spark.createDataFrame(sim_aircraft).write.format("delta").mode("overwrite").saveAsTable("live_aircraft_simulated")
print("Fallback 'live_aircraft_simulated' écrit.")